In [0]:
import pandas as pd
import numpy as np
import re
import os

import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics")

In [0]:
path_saved_designs = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_triple/rfdiffusion/"
folder_designs = [x for x in os.listdir(path_saved_designs) if ("alpha" in x or "beta" in x)]

# Create Master MPNN Dataframe across all runs
list_df_mpnns = []
for folder_design in folder_designs:
  path_design = path_saved_designs + folder_design + "/"
  df_design = pd.read_csv(path_design + "mpnn_results.csv", index_col= 0)
  model_type = "alpha" if "alpha" in folder_design else "beta"
  df_design["rfd_model_type"] = model_type
  df_design["design_folder"] = path_design
  list_df_mpnns.append(df_design)

df_mpnn = pd.concat(list_df_mpnns)
df_mpnn

In [0]:
import biotite
import biotite.structure as struc
from StrucTools import *

def extract_input_motif_coords(path_input_pdb: str, motif_chain_id: str = "C"):
    atom_array_input = extract_atom_array(path_input_pdb)

    # Create input atom mask of motif atoms in the input
    atom_array_input_motif = atom_array_input[atom_array_input.chain_id == motif_chain_id]
    coords_motif_input = atom_array_input_motif.coord
    return coords_motif_input

def extract_design_motif_coords(contigs_str: str, path_pdb_design: str, motif_chain_id: str = "C", binder_chain_id: str = "B"):
    contigs = contigs_str.split(';')
    contig_target, contig_binder = contigs
    res_motif_mask = []
    for region in contig_binder.split('/'):
        if (region[0] == motif_chain_id):
            preserved_region = region[1:]
            start, end = preserved_region.split('-')
            motif_len = int(end) - int(start) + 1
            motif_mask = [True] * motif_len
        else:
            design_len1, design_len2 = region.split('-')
            # Handle case of chains corresponding to those preserved from initial input
            if design_len1 != design_len2:
                design_strand_size = int(design_len2) - int(design_len1[1:]) + 1
            # Handle case of chains corresponding to those designed via RFDiffusion
            elif design_len1 == design_len2:
                design_strand_size = design_len2
            motif_mask = [False] * int(design_strand_size)
        
        res_motif_mask += motif_mask
    
    atom_array_design_complex = extract_atom_array(path_pdb_design)
    atom_array_design_binder = atom_array_design_complex[atom_array_design_complex.chain_id == binder_chain_id]
    mask_atom_motif_design = struc.spread_residue_wise(atom_array_design_binder, res_motif_mask)
    atom_array_design_motif = atom_array_design_binder[mask_atom_motif_design]
    coords_motif_design = atom_array_design_motif.coord
    return coords_motif_design

In [0]:
import biotite.structure as struc
from biotite.structure import superimpose, rmsd
# 1. Extract coordinates of middle strand motif
path_input_pdb = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_triple/input/collagen_triple_helix_alpha2_model_0.pdb"
coords_motif_input = extract_input_motif_coords(path_input_pdb, motif_chain_id= "C")

list_rmsd_middle_strand = []
list_design_paths = []
for index, row in df_mpnn.iterrows():
    # 2. Extract path of the pdb design
    path_pdb_folder = row['design_folder'] + 'all_pdb' + "/"
    path_pdb_design = path_pdb_folder + f"design{row['design']}_n{row['n']}.pdb"
    # 3. Extract contigs as a string
    contigs_str = str(row['contigs'])
    # 4. Extract coordinates of motif in the design
    coords_motif_design = extract_design_motif_coords(contigs_str, path_pdb_design, motif_chain_id= "C")
    # 5. Align input and design motif coordinates
    aligned_design_coords, applied_transformation = superimpose(coords_motif_input, coords_motif_design)
    # 5. Calculate RMSD between input and design
    rmsd_middle_strand = rmsd(coords_motif_input, aligned_design_coords)
    list_rmsd_middle_strand.append(rmsd_middle_strand)
    list_design_paths.append(path_pdb_design)

df_mpnn.insert(loc = 9, column = "rmsd_middle_strand", value = list_rmsd_middle_strand)
df_mpnn['design_pdb_path'] = list_design_paths
df_mpnn.head()



In [0]:
df_mpnn['rmsd_middle_strand'].describe()
df_mpnn['rmsd_middle_strand'].hist(bins= 50)

In [0]:
# Create Kernel Density Plot of MPNN RMSD Motif
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_mpnn["rmsd_motif"], fill=True, ax= ax, label = 'rmsd_motif', alpha = 0.5)
sns.histplot(df_mpnn['rmsd_middle_strand'], fill = True, ax = ax, label = 'rmsd_middle_strand', alpha = 0.5)
plt.title("Triple Helix Motif RMSD from MPNN Produced Sequences")
plt.xlabel("RMSD Motif")
plt.ylabel("Density")
plt.legend(loc = 'upper right')
plt.show()

In [0]:
agg_funcs = ['mean', 'std', 'count']
df_mpnn_copy = df_mpnn.copy()
df_mpnn_copy.rename(columns = {'rmsd_motif' : 'rmsd_triple_helix'}, inplace = True)
df_mpnn_groupedby = df_mpnn_copy.groupby(['rfd_model_type', 'contigs']).agg({'rmsd_middle_strand' : agg_funcs,
                                                                             'rmsd_triple_helix': agg_funcs,
                                                                             'rmsd' : agg_funcs,
                                                                             'plddt' : agg_funcs,
                                                                             'ptm' : agg_funcs,
                                                                             'pae' : agg_funcs
                                                                              }).sort_values([('rmsd_middle_strand', 'mean')], ascending= True)
df_mpnn_groupedby.reset_index(inplace = True)
df_mpnn_groupedby

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(x = df_mpnn['rmsd_motif'], y = df_mpnn['rmsd_middle_strand'], ax = ax, label = 'triple helix vs middle strand')

plt.plot([df_mpnn['rmsd_motif'].min(), df_mpnn['rmsd_motif'].max()],
         [df_mpnn['rmsd_motif'].min(), df_mpnn['rmsd_motif'].max()],
         color='red', linestyle='-', linewidth=2, label='y = x')

plt.title("RMSD Motif vs RMSD Middle Strand")
plt.xlabel("RMSD Motif")
plt.ylabel("RMSD Middle Strand")
plt.legend()
plt.show()

### Visualizing RFDiffusion Designed Structures

In [0]:
df_mpnn_sorted = df_mpnn.sort_values(by = ['rmsd_middle_strand'], ascending = True)
df_mpnn_sorted['rmsd_middle_strand'].describe()

In [0]:
df_mpnn_sorted

In [0]:
import py2Dmol
from StrucTools import visualize_structure
import random
#random_index = random.randint(0, 100)
random_index = 350
print(f"Random index: {random_index}")
print(f"RMSD Middle Strand: {df_mpnn['rmsd_middle_strand'].iloc[random_index]}")
print(f"RMSD: {df_mpnn['rmsd'].iloc[random_index]}")
print(f"RFD Model Type: {df_mpnn['rfd_model_type'].iloc[random_index]}")
print(f"PLDDT: {df_mpnn['plddt'].iloc[random_index]}")
print(f"PTM: {df_mpnn['ptm'].iloc[random_index]}")
print(f"PAE: {df_mpnn['pae'].iloc[random_index]}")
print(f"Contigs: {df_mpnn['contigs'].iloc[random_index]}")
visualize_structure(df_mpnn_sorted.iloc[random_index]['design_pdb_path'])